# KI-Labor: Gedächtnis und Kontext
In diesem Experiment untersuchen wir, wie ein Sprachmodell Informationen verarbeitet. Wir nutzen das Modell **Gemma 3** (1B), das lokal auf dem Rechner über **Ollama** läuft – ohne Internetverbindung und ohne Anmeldung.

**Hinweis:** Das Modell ist ca. 815 MB groß und wird beim ersten Ausführen einmalig heruntergeladen.

---

## Was du nach diesem Notebook weißt:
- Warum ein KI-Chatbot sich normalerweise **nicht** an frühere Nachrichten erinnert
- Was ein **Kontextfenster** ist und wie es sich von menschlichem Gedächtnis unterscheidet
- Wie man durch einen Gesprächsverlauf ein \"Kurzzeitgedächtnis\" **simuliert**
- Was **System-Prompts** sind und wie sie das Verhalten der KI steuern

---

## Voraussetzungen

Damit das Notebook funktioniert, müssen zwei Dinge erledigt sein:

1. **Ollama** muss auf dem Rechner installiert und gestartet sein.
2. Das Python-Paket `ollama` muss installiert sein (nächste Zelle).

Das Paket `ollama` ist die Schnittstelle zwischen diesem Notebook und der lokal laufenden Ollama-Anwendung. Das Modell selbst wird automatisch heruntergeladen, sobald wir es zum ersten Mal aufrufen.

In [ ]:
pip install ollama

In [ ]:
import ollama

MODEL = "gemma3:1b"  # Das Modell, das wir in allen Experimenten verwenden

# Modell herunterladen (nur beim ersten Mal nötig, danach lokal gespeichert)
print(f"Lade Modell '{MODEL}' herunter – das kann beim ersten Mal einige Minuten dauern...")
ollama.pull(MODEL)
print(f"Modell '{MODEL}' ist bereit!")

---

## Teil 1: Das Experiment \"Gedächtnistest\"

Wir senden zwei **getrennte** Nachrichten an die KI. Da wir beim zweiten Mal den Gesprächsverlauf nicht mitschicken, erhält das Modell keinerlei Information aus der ersten Nachricht – es ist, als würde man eine völlig neue Unterhaltung beginnen.

```
 Anfrage 1:  [ "Hi, my name is Max!" ]        → KI antwortet
                       ↓
               wird NICHT weitergegeben
                       ↓
 Anfrage 2:  [ "What is my name?" ]            → KI weiß es nicht mehr
```

In [ ]:
# 1. Nachricht: Wir stellen uns vor
chat1 = [{"role": "user", "content": "Hi, my name is Max!"}]
out1 = ollama.chat(model=MODEL, messages=chat1)
print("KI Antwort 1:", out1['message']['content'])

print("\n--- NEUE ANFRAGE (kein Verlauf übergeben) ---\n")

# 2. Nachricht: Wir fragen nach unserem Namen – ohne den vorherigen Verlauf
chat2 = [{"role": "user", "content": "What is my name?"}]
out2 = ollama.chat(model=MODEL, messages=chat2)
print("KI Antwort 2:", out2['message']['content'])

### Reflexion, bevor du weitermachst:
Beantworte diese Fragen kurz für dich (oder diskutiere sie mit deiner Sitznachbarin / deinem Sitznachbarn):

1. Warum kennt die KI den Namen nicht mehr? Handelt es sich dabei um einen 'Fehler' des Modells?
2. Wie funktioniert das bei ChatGPT oder anderen Chatbots – haben die wirklich ein 'Gedächtnis'?  
   *Hinweis: Überlege, ob ChatGPTs „Memory"-Funktion dasselbe ist wie das Kontextfenster – oder ob da etwas anderes dahintersteckt.*
3. Was müsste technisch passieren, damit die KI den Namen beim zweiten Aufruf noch kennt?

---

## Teil 2: Die Lösung – Gedächtnis-Simulation über den Gesprächsverlauf

Sprachmodelle haben kein echtes Gedächtnis. Sie verarbeiten immer nur den Text, der ihnen **in diesem Moment** übergeben wird – den sogenannten **Kontext**. Wenn wir wollen, dass die KI 'weiß', was vorher gesagt wurde, müssen wir ihr den gesamten bisherigen Gesprächsverlauf als Liste mitgeben.

```
 Anfrage 1:  [ "Hi, my name is Max..." ]              → KI antwortet
             [ "Hello Max! Nice to meet you!" ]       ↘ wird gespeichert
                                                       ↓
 Anfrage 2:  [ "Hi, my name is Max..."         ]      ↗
             [ "Hello Max! Nice to meet you!"  ]
             [ "What is my name?"              ]      → KI kennt den Namen!
```

Jede Nachricht in der Liste hat zwei Felder:
- `"role"`: Wer hat diese Nachricht geschrieben? – `"user"`, `"assistant"` oder `"system"`
- `"content"`: Der eigentliche Text der Nachricht

In [ ]:
# Der gesamte Gesprächsverlauf wird als Liste übergeben.
# Jede Nachricht hat eine 'role' (user oder assistant) und einen 'content' (den Text).
verlauf = [
    {"role": "user",      "content": "Hi, my name is Max and I love sports!"},
    {"role": "assistant", "content": "Hello Max! It is nice to meet a sports enthusiast. What kind of sports do you like?"},
    {"role": "user",      "content": "What is my name and what do I like?"}
]

output = ollama.chat(model=MODEL, messages=verlauf)
print("KI Antwort mit Gedächtnis:")
print(output['message']['content'])

> **Was ist ein Token?** Sprachmodelle lesen Text nicht Buchstabe für Buchstabe, sondern in sogenannten *Tokens* – grob gesagt Wortfragmenten. Das Wort „Informatik" zählt z. B. als 1–2 Tokens, ein ganzer Satz als ~10–20. Das Kontextfenster ist in Tokens begrenzt, nicht in Wörtern oder Zeichen. Das erklärt, warum die Grenze, ab der ältere Informationen „herausfallen", schwer vorherzusagen ist – und warum sehr lange Gespräche irgendwann unzuverlässig werden.

---

## Übung:

**Schritt 1 (Pflicht):** Ändere `verlauf_uebung` so ab, dass das Sprachmodell eine bestimmte Rolle einnimmt (z.B. *a grumpy Bavarian*, *a robot from the future*, *a funny pirate*, *a strict teacher* oder *a medieval scholar*). Füge dazu ganz oben in der Liste eine **System-Nachricht** ein:
```python
{"role": "system", "content": "You are a grumpy Bavarian."}
```

**Schritt 2 (Erweiterung):** Verlängere das Gespräch um mindestens 2 weitere Nachrichten (`"user"` und `"assistant"` abwechselnd). Bleibt die KI dabei in ihrer Rolle?

**Schritt 3 (Bonus):** Schreibe eine Schleife, die den Gesprächsverlauf automatisch aufbaut. Starte mit einer leeren Liste und einer System-Nachricht. Füge nach jeder KI-Antwort die Antwort mit `role: "assistant"` an den Verlauf an, bevor du die nächste Benutzernachricht hinzufügst.

Hier ist ein Gerüst als Ausgangspunkt – was fehlt noch?
```python
system_nachricht = {"role": "system", "content": "You are a ..."}
verlauf = [system_nachricht]

nachrichten = [
    "Hello! Who are you?",
    "What do you think about pizza?",
    "Tell me a short joke.",
]

for nachricht in nachrichten:
    verlauf.append({"role": "user", "content": nachricht})
    output = ollama.chat(model=MODEL, messages=verlauf)
    antwort = output['message']['content']
    print("KI:", antwort)
    print("---")
    verlauf.append(...)  # Was kommt hier hin?
```

**Reflexion:** Was sind die Grenzen dieses simulierten Gedächtnisses? Was passiert technisch, wenn ein Gespräch sehr lang wird?

In [ ]:
# Hier ist Platz für deine Lösung (Schritt 1 & 2):
verlauf_uebung = [
    # {"role": "system",    "content": "You are ..."},
    # {"role": "user",      "content": "..."},
    # {"role": "assistant", "content": "..."},
    # {"role": "user",      "content": "..."},
]

output_uebung = ollama.chat(model=MODEL, messages=verlauf_uebung)
print(output_uebung['message']['content'])

---

## Fazit

Was wir heute gesehen haben, gilt für **alle** großen Sprachmodelle, auch für ChatGPT, Gemini oder Copilot:

- Sie haben **kein echtes Langzeitgedächtnis**. Bei jeder Anfrage erhält das Modell den gesamten bisherigen Gesprächsverlauf als Texteingabe – das ist das sogenannte **Kontextfenster**.
- Dieses Kontextfenster ist **begrenzt**: Bei älteren Modellen auf einige Tausend, bei modernen auf Millionen von Tokens. Wenn das Fenster voll ist, fällt älterer Gesprächsverlauf heraus – die KI wird 'vergesslicher'.
- **System-Prompts** bestimmen, wie sich die KI verhält: welche Rolle sie einnimmt, welchen Ton sie verwendet oder was sie nicht tun soll. Jeder Dienst, der KI einsetzt, nutzt solche Prompts im Hintergrund.
- **Ollama** ermöglicht es, Sprachmodelle vollständig **lokal** zu betreiben – keine Daten werden an externe Server gesendet. Das ist besonders relevant, wenn man bedenkt, was bei kommerziellen Diensten mit den eingegebenen Inhalten passiert.

Das erklärt, warum Chatbots manchmal inkonsistent wirken: Es gibt kein Bewusstsein und kein Gedächtnis – nur Textverarbeitung auf Basis dessen, was im aktuellen Kontext steht.